# 🎮 Module 2 — Class 4: Pandas — Filtering, Grouping, and Merging

**Lecture:** [https://bepro-aiml.github.io/aiml-platform/#/module/2/class/4](https://bepro-aiml.github.io/aiml-platform/#/module/2/class/4)

## 🧠 Pandas = a Super-Powered Spreadsheet

Imagine Excel — but instead of clicking and dragging, you tell it exactly what you want with one line of code, and it answers in milliseconds even on millions of rows.

That's **Pandas**.

Today's mission: you are a mission control commander. Agents are running missions across the globe — Satellite Analysis, Robot Training, Drone Flight. Each mission earns a **Success Score** out of 100. Your job is to use Pandas to find patterns: who's crushing it, which mission is hardest, and which agents need backup.

By the end of this notebook you'll know how to:
- 🔍 **Filter** rows that match conditions
- 📊 **Group** data and summarise it
- 🔗 **Merge** two tables by a shared column
- 📈 **Visualise** the result

👾 *Game on.*

---
## 1. 🛰️ Generate the Mission Data

We'll create a synthetic dataset called `mission_data.csv` with 4 columns:
- `Agent_Name` — the operator running the mission
- `Mission_Type` — Satellite Analysis / Robot Training / Drone Flight
- `Success_Score` — a number between 0 and 100
- `Region` — Tashkent, Sydney, or London

Run the cell below to generate the file and load it into a DataFrame.

In [ ]:
import pandas as pd
import numpy as np

# Reproducible randomness — everyone gets the same data
rng = np.random.default_rng(42)

agents = [
    'Ali', 'Zara', 'Jasur', 'Madina', 'Otabek',
    'Liam', 'Mia', 'Noah', 'Olivia', 'Ethan',
    'Aisha', 'Tom', 'Layla', 'Hugo', 'Nodira'
]
missions = ['Satellite Analysis', 'Robot Training', 'Drone Flight']
regions = ['Tashkent', 'Sydney', 'London']

n = 40
df = pd.DataFrame({
    'Agent_Name': rng.choice(agents, n),
    'Mission_Type': rng.choice(missions, n),
    'Success_Score': rng.integers(40, 100, n),
    'Region': rng.choice(regions, n)
})

df.to_csv('mission_data.csv', index=False)
df.head(10)

### ✅ Check Your Understanding

1. Run `df.shape` in the next cell. How many rows and columns are there?
2. Run `df.dtypes`. Which column is the only numeric one?
3. In one sentence — what does `df.to_csv('mission_data.csv', index=False)` do, and what would `index=True` change?

In [ ]:
# Your inspection code here
print(df.shape)
print(df.dtypes)

---
## 2. 🔍 Task 1 — Filtering: Find the Tashkent Stars

**Mission brief:** Headquarters wants to know which agents in **Tashkent** scored **above 80**. They're the candidates for the next high-stakes mission.

In Pandas, you filter rows using **boolean conditions**. Combine them with `&` (AND) and `|` (OR) — and **always wrap each condition in parentheses**.

```python
df[(df.col_a >= 18) & (df.col_b == 'something')]
```

🚨 *Common bug:* using `and` instead of `&` raises `ValueError: The truth value of a Series is ambiguous`. Pandas needs the bitwise version.

In [ ]:
tashkent_stars = df[(df['Region'] == 'Tashkent') & (df['Success_Score'] > 80)]
tashkent_stars

### ✅ Check Your Understanding

1. How many Tashkent agents scored above 80?
2. Modify the filter to find **Sydney** agents who scored *below* 60. Write the code in the next cell and a one-sentence interpretation in markdown afterwards.
3. *Tricky:* what happens if you write `df[df.Region == 'Tashkent' & df.Success_Score > 80]` (no parentheses)? Try it — explain the error in your own words.

In [ ]:
# Your filter for Sydney + below 60 here

# Then try the buggy unparenthesised version and read the traceback:
# df[df.Region == 'Tashkent' & df.Success_Score > 80]

**Your interpretation:**

---
## 3. 📊 Task 2 — GroupBy: Which Mission Is the Hardest?

**Mission brief:** Command wants to know which mission type is the most brutal — the one with the **lowest average success score**. That's where to invest more training.

`groupby` works in three steps: **split** the data into buckets, **apply** a function to each bucket, **combine** the results.

```python
df.groupby('column_to_split_by')['column_to_summarise'].mean()
```

💡 *Mental model:* think of it as Excel's `SUMIF`/`AVERAGEIF` — but for any function and any column.

In [ ]:
avg_score_by_mission = df.groupby('Mission_Type')['Success_Score'].mean().round(2)
avg_score_by_mission.sort_values()

### ✅ Check Your Understanding

1. Which mission type has the **lowest** average score (i.e. the hardest)?
2. What does `.sort_values()` do? What would `.sort_values(ascending=False)` do instead?
3. **Type check:** what does `df.groupby('Mission_Type')['Success_Score'].mean()` return — DataFrame, Series, list, or scalar? Run `type(...)` on it to verify your guess.
4. **Stretch:** use `.agg()` with a dict to compute, in one call, the mean *and* the count of missions per `Mission_Type`. Which mission was attempted the most often?

In [ ]:
# Type check + stretch challenge here
print(type(avg_score_by_mission))

# Stretch:
# df.groupby('Mission_Type').agg({'Success_Score': ['mean', 'count']})

---
## 4. 🔗 Task 3 — Merging: Add Experience Levels

**Mission brief:** Each agent has an experience level (Junior / Senior / Veteran). It lives in a *separate* table. To analyse score-by-experience, we need to **merge** the two tables on the shared `Agent_Name` column.

```python
pd.merge(left_df, right_df, on='shared_column', how='left')
```

Join types you need to know:
- `inner` — only rows where the key exists in BOTH tables
- `left` — keep all rows from the left, fill `NaN` if no match on the right
- `right` — keep all rows from the right
- `outer` — keep everything from both

In [ ]:
experience = pd.DataFrame({
    'Agent_Name': ['Ali', 'Zara', 'Jasur', 'Madina', 'Otabek',
                   'Liam', 'Mia', 'Noah', 'Olivia', 'Ethan'],
    'Experience_Level': ['Junior', 'Senior', 'Veteran', 'Junior', 'Senior',
                         'Veteran', 'Junior', 'Senior', 'Veteran', 'Junior']
})
experience

In [ ]:
merged = pd.merge(df, experience, on='Agent_Name', how='left')
merged.head(10)

### ✅ Check Your Understanding

1. Notice we used `how='left'`. Some agents in `df` are not in the `experience` table (e.g. 'Aisha', 'Tom'). What appears in their `Experience_Level` column after the merge?
2. Re-run the merge with `how='inner'`. How many rows did the result lose, and **why**?
3. **Stretch:** chain together what you learned today — group the *merged* DataFrame by `Experience_Level` and compute the mean `Success_Score`. Do veterans actually score higher than juniors?

In [ ]:
# Your inner-merge experiment

# Stretch: groupby experience level and check mean score


**Your answer:**

---
## 5. 📈 Visualise the Scoreboard

Numbers in a table are fine for code — but for *humans*, a chart wins every time.

Below: a simple bar chart showing the **average success score per mission type**. The lowest bar = the hardest mission.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
avg_score_by_mission.sort_values().plot(kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#2ecc71'])
ax.set_title('🎯 Average Success Score by Mission Type', fontsize=14)
ax.set_xlabel('Mission Type')
ax.set_ylabel('Average Score (0–100)')
ax.set_ylim(0, 100)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### ✅ Check Your Understanding (final)

1. **Read the chart.** Which mission type is the easiest? Which is the hardest? By how many points does the easiest beat the hardest?
2. **Critique it.** Pick one of the *Five Rules for Honest Charts* the lecture taught you. Does this chart pass that rule? Why or why not?
3. **Improve it.** Make a *second* bar chart — this time grouped by **Region**. Add a clear title, axis labels, and a sentence underneath in markdown explaining the most interesting thing the chart reveals.

In [ ]:
# Your improved bar chart by Region here


**What this second chart reveals:**

---
## 🏁 Mission Debrief

You now have all four core Pandas operations under your belt:

| Operation | Pandas function | What it answers |
| --- | --- | --- |
| 🔍 Filter | `df[(df.col == x) & (df.col2 > y)]` | *Which rows match my condition?* |
| 📊 GroupBy | `df.groupby('col')['x'].mean()` | *What's the per-group summary?* |
| 🔗 Merge | `pd.merge(a, b, on='key', how='left')` | *How do I combine two tables?* |
| 📈 Plot | `series.plot(kind='bar')` | *How do I show this to humans?* |

### Submit
Save this notebook as `Module2_Class4_<YourName>.ipynb` and submit a PR to your group repo at `module-2/class_4/submissions/<YourName>/`.

🎮 *See you in Class 5 — Visualization.*